# 58. 回归图（regplot / lmplot）

<!-- module-learning-arc:start -->
> **Seaborn 模块主线｜第 15 / 20 步：探索变量关系与趋势**
>
> **持续应用背景：** 开展客群消费行为差异研究：先固定样本和统计语义，再比较分布、关系和分面结果，判断差异是否稳定。
>
> **承接上一阶段：** 统计折线图（lineplot）  →  **本章任务：** 回归图（regplot / lmplot）  →  **下一步：** 联合分布图（jointplot）
>
> **大作业连接：** 本章练习将成为《客群消费行为差异研究》的一部分，最终需要从样本口径和分布比较走到关系验证、分面研究与因果边界说明。
<!-- module-learning-arc:end -->


## 本章场景

手头有两列都是数值的数据，比如网站的访问量和当天的销售额，单独看任何一列都看不出它们"一起"变化的规律。



## 本章目标

学完本章，你将能够：

- **理解**：理解「回归图（regplot / lmplot）」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「回归图（regplot / lmplot）」的关键输出指标。
- **迁移**：能把「回归图（regplot / lmplot）」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 58.1 适用场景

**背景引入**：手头有两列都是数值的数据，比如网站的访问量和当天的销售额，单独看任何一列都看不出它们"一起"变化的规律。回归图在散点图的基础上，沿着所有数据点最贴合的走向画出一条拟合线，让你一眼看清两个量到底是一起涨还是一起跌、关系是直是弯。学会它，你在做业务分析时就能快速判断两个指标间的依赖方向，而不必把因果论证留到正式建模阶段。

打个比方：regplot 像'给全班同学的身高体重画一条最佳趋势线'——它不保证穿过每一个点，而是找一条'离所有点整体最近'的直线，让你一眼看出两个量是随大流一起涨还是一起跌。它描述的是'一起变化的走向'，不等于'谁导致了谁'。

需要描述两个数值变量的拟合方向，而不是证明因果。


## 58.2 数据结构

两列数值；lmplot可增加分类分组和分面。


## 58.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 order=1 改为 order=2，观察线性与二次多项式拟合的曲线差异
2. 修改 ci=95 为 ci=None，对比显示与隐藏置信区间的视觉效果
3. 添加 robust=True 参数，说明稳健拟合对异常点的抗性


## 58.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `plt.subplots()`、`sns.regplot()`、`ax.set()`、`fig.tight_layout()` | 需要描述两个数值变量的拟合方向，而不是证明因果。 | 把回归线解释为因果 |
| 进阶变体 | `sns.lmplot()`、`grid.set_axis_labels()`、`grid.set_titles()`、`grid.fig.suptitle()` | 在基础图表上增加分组、注释、布局或交互 | 忽略非线性和异方差 |
| 关键参数 | `order` | 多项式阶数 | 把回归线解释为因果 |
| 关键参数 | `robust` | 稳健拟合 | 忽略非线性和异方差 |
| 关键参数 | `ci` | 区间 | 只展示拟合线不展示原始点 |
| 关键参数 | `scatter_kws/line_kws` | 样式 | 把回归线解释为因果 |


## 58.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-58 -->
### 数学推导｜最小二乘回归线

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜定义预测与残差。** $\hat y_i=\beta_0+\beta_1x_i$，$e_i=y_i-\hat y_i$。

**第 2 步｜让残差平方和最小。** 对 $SSE=\sum_ie_i^2$ 分别对 $\beta_0$、$\beta_1$ 求导并令其为 0，可得

$$
\hat\beta_1=\frac{\sum_i(x_i-\bar x)(y_i-\bar y)}{\sum_i(x_i-\bar x)^2},
\qquad
\hat\beta_0=\bar y-\hat\beta_1\bar x
$$

**第 3 步｜代回得到拟合线。** 斜率本质上是“共同变化”除以 $x$ 自身变化。

**把上面的关系收束为本章计算式：**

$$
\hat{y}=\beta_0+\beta_1x,\qquad \min_{\beta_0,\beta_1}\sum_i(y_i-\hat{y}_i)^2
$$

**符号解释：** $\beta_1$ 描述 $x$ 每增加 1 单位时预测均值的线性变化。

**代码对应：** `regplot`/`lmplot` 展示拟合关系；同时检查残差、异常点与分组。

**使用边界：** 回归线描述条件关联，不自动提供因果解释。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，即使 seaborn 的 sns.set_theme
#      会重置字体，运行时也会在 set_theme 之后自动恢复。因此这里无需手动
#      import 或 addfont，直接使用即可。

# 1️⃣ 主题与数据导入：统一画风，读取三个公开数据集
sns.set_theme(style="whitegrid", context="notebook")

diamonds = pd.read_csv("/datasets/diamonds.csv")
taxis = pd.read_csv("/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
flights = pd.read_csv("/datasets/flights.csv")
print(
    f"Diamonds {
        len(diamonds):,    } | Taxis {
            len(taxis):,        } | Flights {
                len(flights):,            } 行"
)


In [ ]:
# 2️⃣ 特征工程：把原始字段映射成图表统一使用的列名与派生指标
orders_full = diamonds.assign(
    category=diamonds["cut"],
    channel=diamonds["color"],
    region=diamonds["clarity"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    satisfied=np.where(
        diamonds["price"] >= diamonds["price"].median(),
        "高于中位价",
        "不高于中位价",
    ),
)
orders = orders_full.sample(2_000, random_state=36)

marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"),
    visits=taxis["distance"],
    ad_spend=taxis["tip"],
    sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(
    min(2_000, len(marketing_full)), random_state=36
).copy()

daily = flights.assign(
    date=pd.to_datetime(
        flights["year"].astype(str) + "-" + flights["month"] + "-01"
    ),
    region="AirPassengers",
    sales=flights["passengers"],
)
print(
    f"样本：orders {
        len(orders):,    } | marketing {
            len(marketing):,        } | daily {
                len(daily):,            } 行"
)


## 58.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.regplot(
    data=marketing,
    x="visits",
    y="sales",
    scatter_kws={"alpha": 0.45, "s": 25},
    line_kws={"color": "#d93025"},
    ax=ax,
)
ax.set(title="访问量与销售额线性趋势", xlabel="访问量", ylabel="销售额")
fig.tight_layout()
plt.show()


**练一练**：把 52.4 基础图表里的回归拟合稍微改动一下。把 `order` 参数从线性（1）改成二次（2），再看看拟合曲线怎么随数据点弯曲；同时把散点透明度 `scatter_kws["alpha"]` 改小一点（如 0.3），观察重叠散点变得浅色后的效果。运行并核对自检打印，体会这个参数对不同拟合形状的调节作用。


In [ ]:
# 请在下方填写代码：完成下方填空，复现并微调 52.4 的基础回归图。


In [ ]:
# 讲解：order 控制多项式拟合的次数，order=2 会用一条二次曲线去贴合数据，
# 更适合捕捉下凹或上翘的趋势；alpha 控制散点透明度，越小越能看清重叠点。

order_val = 2
alpha_val = 0.3

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.regplot(
    data=marketing,
    x="visits",
    y="sales",
    order=order_val,
    scatter_kws={"alpha": alpha_val, "s": 25},
    ax=ax,
)
ax.set(title="访问量与销售额：二次拟合", xlabel="访问量", ylabel="销售额")
fig.tight_layout()
plt.show()


## 58.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

grid = sns.lmplot(
    data=marketing,
    x="visits",
    y="sales",
    hue="channel",
    col="channel",
    col_wrap=3,
    height=3.2,
    scatter_kws={"alpha": 0.4, "s": 22},
    palette="colorblind",
)
grid.set_axis_labels("访问量", "销售额")
grid.set_titles("{col_name}")
grid.fig.suptitle("分渠道回归趋势", y=1.04)
plt.show()


## 58.8 参数说明

- order：多项式阶数
- robust：稳健拟合
- ci：区间
- scatter_kws/line_kws：样式


## 58.9 结果解读

读取斜率方向、散点离散和区间；检查异常点是否主导拟合。


## 58.10 本章实训：分组比较与不确定性

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="region", y="sales", ci=None, ax=ax, color="#0F766E"
)
ax.set_title("地区销售额比较")
ax.set_ylabel("销售额")
plt.show()


### 58.10.1 第一个结果怎么读

Seaborn 负责把 DataFrame 的字段映射为图形编码；先明确横轴、纵轴和每行数据的粒度，再选择图表。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
report = report.sort_values("sales", ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="sales", y="region", ci=None, ax=ax, color="#F59E0B"
)
ax.set_title("按销售额排序的地区比较")
ax.set_xlabel("销售额")
ax.set_ylabel("地区")
plt.show()


### 58.10.2 第二个结果怎么读

第二个实验只改变排序和坐标方向，让读者更容易找到最大值。图表调整必须服务于阅读任务。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 58.11 错误恢复：分组字段缺失怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
required = {"region", "sales"}
missing = required - set(report.columns)
if missing:
    print("缺少字段：", sorted(missing))
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.barplot(data=report, x="region", y="sales", ci=None, ax=ax)
    ax.set_title("地区销售额")
    plt.show()


### 58.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

绘图前先检查字段是否存在。把字段检查放在画图之前，错误会更接近真正原因，也更容易恢复。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 58.12 易错点提醒

- 把回归线解释为因果
- 忽略非线性和异方差
- 只展示拟合线不展示原始点


## 58.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 58.14 独立迁移练习

修改一个分组、排序或统计设置，并比较修改前后的结论。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：把线性回归换成稳健回归，观察抗离群点的差异
# 【目标】稳健回归(robust=True)对离群点不敏感，练习体会其与普通回归的差别。
import matplotlib.pyplot as plt
import seaborn as sns

# 起点示例(已可运行)：加 robust=True，回归线更少受离群点拉动。
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.regplot(
    data=marketing,
    x="visits",
    y="sales",
    robust=True,
    scatter_kws={"alpha": 0.45, "s": 25},
    line_kws={"color": "#188038"},
    ax=ax,
)
ax.set(title="访问量与销售额稳健回归", xlabel="访问量", ylabel="销售额")
fig.tight_layout()
plt.show()

# ---- 反思记录：稳健回归的线与普通回归有何不同 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.regplot(
    data=marketing,
    x="ad_spend",
    y="sales",
    order=2,
    scatter_kws={"alpha": 0.4, "s": 24},
    line_kws={"color": "#188038"},
    ax=ax,
)
ax.set(
    title="广告投入与销售额的非线性趋势检查",
    xlabel="广告投入",
    ylabel="销售额",
)
fig.tight_layout()
plt.show()


## 58.15 小结

通过regplot和lmplot叠加回归趋势，检查线性关系和分组差异。


### 58.15.1 你已经掌握

- 判断回归图（regplot / lmplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 58.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `order` | 多项式阶数 |
| `robust` | 稳健拟合 |
| `ci` | 区间 |
| `scatter_kws/line_kws` | 样式 |


### 58.15.3 需要注意

- 把回归线解释为因果
- 忽略非线性和异方差
- 只展示拟合线不展示原始点


### 58.15.4 完成检查

- [ ] 能判断什么问题适合使用回归图（regplot / lmplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 58.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
